# 🤖 Modelado · Clasificación Multiclase de Obesidad

### Proyecto Grupal — Clasificación Multiclase · p7_g1_multiclase

**Autor:** JJ  
**Fecha:** 2026-03-18  
**Input:** `data/processed/` (generado por `Preprocesamiento_p7_g1_multiclase_JJ.ipynb`)  
**Target:** `NObeyesdad` — 7 niveles de peso corporal

---

## 🗺️ Hoja de Ruta

| Bloque | Contenido |
|--------|-----------|
| **0** | Imports y carga de datos |
| **1** | Baseline: Decision Tree |
| **2** | Random Forest |
| **3** | KNN |
| **4** | SVM |
| **5** | XGBoost |
| **6** | Comparativa final de modelos |
| **7** | Mejor modelo: análisis detallado |
| **8** | Guardar modelo ganador |

> ⚠️ **Requisito previo:** ejecuta primero `Preprocesamiento_p7_g1_multiclase_JJ.ipynb` para generar los CSVs y PKLs en `data/processed/` y `models/`.

---

### 🏷️ Clases del target (confirmadas en preprocesamiento)

| Código | Etiqueta |
|--------|----------|
| 0 | Insufficient_Weight |
| 1 | Normal_Weight |
| 2 | Overweight_Level_I |
| 3 | Overweight_Level_II |
| 4 | Obesity_Type_I |
| 5 | Obesity_Type_II |
| 6 | Obesity_Type_III |

---

## 📦 BLOQUE 0 — Imports y carga de datos

In [1]:
# ─── Librerías ────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os, warnings
warnings.filterwarnings('ignore')

# Modelos
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Evaluación
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.model_selection import cross_val_score, GridSearchCV

print('✅ Librerías cargadas')

ModuleNotFoundError: No module named 'xgboost'

In [ ]:
# ─── Cargar datos del preprocesamiento ────────────────────────────────────────
X_train        = pd.read_csv('../data/processed/X_train.csv')
X_test         = pd.read_csv('../data/processed/X_test.csv')
X_train_scaled = pd.read_csv('../data/processed/X_train_scaled.csv')
X_test_scaled  = pd.read_csv('../data/processed/X_test_scaled.csv')
y_train        = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test         = pd.read_csv('../data/processed/y_test.csv').squeeze()

# Label encoder para recuperar nombres de clases
le = joblib.load('../models/label_encoder_target.pkl')
class_names = le.classes_
# Confirmado en preprocesamiento:
# [Insufficient_Weight, Normal_Weight, Overweight_Level_I, Overweight_Level_II,
#  Obesity_Type_I, Obesity_Type_II, Obesity_Type_III]

print(f'✅ Datos cargados')
print(f'   X_train : {X_train.shape}  |  X_test : {X_test.shape}')
print(f'   y_train : {y_train.shape}  |  y_test : {y_test.shape}')
print(f'\n   Columnas ({len(X_train.columns)}): {list(X_train.columns)}')
print(f'\n   Clases ({len(class_names)}):')
for i, c in enumerate(class_names):
    print(f'     {i} → {c}')

In [ ]:
# ─── Función de evaluación reutilizable ───────────────────────────────────────
resultados = {}  # dict global para la comparativa final

def evaluar_modelo(nombre, modelo, X_tr, X_te, y_tr, y_te, cv=5):
    """Entrena, evalúa con CV y test, imprime métricas y guarda en resultados."""
    modelo.fit(X_tr, y_tr)
    y_pred    = modelo.predict(X_te)
    acc_test  = accuracy_score(y_te, y_pred)
    cv_scores = cross_val_score(modelo, X_tr, y_tr, cv=cv, scoring='accuracy')

    resultados[nombre] = {
        'modelo'  : modelo,
        'acc_test': acc_test,
        'cv_mean' : cv_scores.mean(),
        'cv_std'  : cv_scores.std(),
        'y_pred'  : y_pred
    }

    sep = '=' * 60
    print(f'\n{sep}')
    print(f'  {nombre}')
    print(sep)
    print(f'  Accuracy test  : {acc_test:.4f}')
    print(f'  CV ({cv}-fold)   : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
    print(f'\n{classification_report(y_te, y_pred, target_names=class_names)}')
    return modelo

print('✅ Función evaluar_modelo() lista')

---

## 🌳 BLOQUE 1 — Baseline: Decision Tree

**¿Por qué empezar con un árbol?**  
Es el modelo más interpretable y rápido. Nos da el **baseline mínimo** que el resto deben superar.  
No necesita escalado → trabaja directamente con `X_train` / `X_test`.

In [ ]:
# ─── Decision Tree · sin restricción (baseline) ───────────────────────────────
dt = DecisionTreeClassifier(random_state=42)
evaluar_modelo('Decision Tree (baseline)', dt, X_train, X_test, y_train, y_test)

In [ ]:
# ─── Decision Tree · curva overfitting vs max_depth ───────────────────────────
depths = range(2, 21)
acc_tr_list, acc_te_list = [], []

for d in depths:
    clf = DecisionTreeClassifier(max_depth=d, random_state=42)
    clf.fit(X_train, y_train)
    acc_tr_list.append(accuracy_score(y_train, clf.predict(X_train)))
    acc_te_list.append(accuracy_score(y_test,  clf.predict(X_test)))

plt.figure(figsize=(9, 4))
plt.plot(depths, acc_tr_list, label='Train', marker='o')
plt.plot(depths, acc_te_list, label='Test',  marker='s')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Decision Tree — Overfitting vs max_depth')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_depth = list(depths)[np.argmax(acc_te_list)]
print(f'\n🏆 Mejor max_depth en test: {best_depth}  →  acc = {max(acc_te_list):.4f}')

In [ ]:
# ─── Decision Tree · modelo ajustado ──────────────────────────────────────────
dt_tuned = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
evaluar_modelo('Decision Tree (tuned)', dt_tuned, X_train, X_test, y_train, y_test)

---

## 🌲 BLOQUE 2 — Random Forest

**¿Por qué Random Forest?**  
Ensemble de árboles con bagging. Reduce el overfitting del árbol individual.  
Suele ser el modelo de referencia para datos tabulares de tamaño medio.  
No necesita escalado.

In [ ]:
# ─── Random Forest · base ─────────────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
evaluar_modelo('Random Forest (base)', rf, X_train, X_test, y_train, y_test)

In [ ]:
# ─── Random Forest · Feature Importance ───────────────────────────────────────
rf_trained = resultados['Random Forest (base)']['modelo']
importances = pd.Series(
    rf_trained.feature_importances_,
    index=X_train.columns
).sort_values(ascending=True)

plt.figure(figsize=(8, 6))
importances.plot(kind='barh', color='steelblue')
plt.title('Random Forest — Feature Importance (Gini)')
plt.xlabel('Importancia media')
plt.tight_layout()
plt.show()

print('\nTop 5 features:')
print(importances.sort_values(ascending=False).head())

In [ ]:
# ─── Random Forest · GridSearchCV ─────────────────────────────────────────────
param_grid_rf = {
    'n_estimators'    : [100, 200],
    'max_depth'       : [None, 10, 20],
    'min_samples_split': [2, 5]
}

gs_rf = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid_rf, cv=5, scoring='accuracy', n_jobs=-1, verbose=1
)
gs_rf.fit(X_train, y_train)

print(f'\n🏆 Mejores params RF : {gs_rf.best_params_}')
print(f'   CV score          : {gs_rf.best_score_:.4f}')

evaluar_modelo('Random Forest (tuned)', gs_rf.best_estimator_,
               X_train, X_test, y_train, y_test)

---

## 📍 BLOQUE 3 — KNN

**¿Por qué KNN?**  
Modelo no paramétrico basado en distancia euclidiana.  
**Muy sensible al escalado** → usa `X_train_scaled` / `X_test_scaled`.  
El hiperparámetro clave es `k` (n_neighbors).

In [ ]:
# ─── KNN · búsqueda de k óptimo ───────────────────────────────────────────────
k_range = range(1, 31)
acc_k   = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
    knn.fit(X_train_scaled, y_train)
    acc_k.append(accuracy_score(y_test, knn.predict(X_test_scaled)))

plt.figure(figsize=(9, 4))
plt.plot(list(k_range), acc_k, marker='o', color='darkorange')
plt.xlabel('k (n_neighbors)')
plt.ylabel('Accuracy test')
plt.title('KNN — Accuracy vs k')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_k = list(k_range)[np.argmax(acc_k)]
print(f'\n🏆 Mejor k: {best_k}  →  acc = {max(acc_k):.4f}')

In [ ]:
# ─── KNN · modelo con mejor k ─────────────────────────────────────────────────
knn_tuned = KNeighborsClassifier(n_neighbors=best_k, n_jobs=-1)
evaluar_modelo('KNN (k óptimo)', knn_tuned,
               X_train_scaled, X_test_scaled, y_train, y_test)

---

## ⚡ BLOQUE 4 — SVM

**¿Por qué SVM?**  
Busca el hiperplano de máximo margen. Potente en espacios de alta dimensión.  
**Requiere escalado** → usa `X_train_scaled` / `X_test_scaled`.  
Usamos kernel RBF, que suele funcionar bien en datos tabulares.

In [ ]:
# ─── SVM · base ───────────────────────────────────────────────────────────────
svm = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
evaluar_modelo('SVM (base)', svm,
               X_train_scaled, X_test_scaled, y_train, y_test)

In [ ]:
# ─── SVM · GridSearchCV ───────────────────────────────────────────────────────
# ⏱️ Puede tardar 3-5 min dependiendo del equipo
param_grid_svm = {
    'C'     : [0.1, 1, 10, 100],
    'gamma' : ['scale', 'auto', 0.01, 0.1],
    'kernel': ['rbf']
}

gs_svm = GridSearchCV(
    SVC(random_state=42),
    param_grid_svm, cv=5, scoring='accuracy', n_jobs=-1, verbose=1
)
gs_svm.fit(X_train_scaled, y_train)

print(f'\n🏆 Mejores params SVM : {gs_svm.best_params_}')
print(f'   CV score           : {gs_svm.best_score_:.4f}')

evaluar_modelo('SVM (tuned)', gs_svm.best_estimator_,
               X_train_scaled, X_test_scaled, y_train, y_test)

---

## 🚀 BLOQUE 5 — XGBoost

**¿Por qué XGBoost?**  
Gradient boosting optimizado. Estado del arte en datasets tabulares estructurados.  
No necesita escalado. Maneja bien clases desequilibradas con `scale_pos_weight`.

In [ ]:
# ─── XGBoost · base ───────────────────────────────────────────────────────────
xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)
evaluar_modelo('XGBoost (base)', xgb, X_train, X_test, y_train, y_test)

In [ ]:
# ─── XGBoost · GridSearchCV ───────────────────────────────────────────────────
param_grid_xgb = {
    'n_estimators' : [100, 200, 300],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth'    : [4, 6, 8]
}

gs_xgb = GridSearchCV(
    XGBClassifier(eval_metric='mlogloss', random_state=42, n_jobs=-1),
    param_grid_xgb, cv=5, scoring='accuracy', n_jobs=-1, verbose=1
)
gs_xgb.fit(X_train, y_train)

print(f'\n🏆 Mejores params XGBoost : {gs_xgb.best_params_}')
print(f'   CV score              : {gs_xgb.best_score_:.4f}')

evaluar_modelo('XGBoost (tuned)', gs_xgb.best_estimator_,
               X_train, X_test, y_train, y_test)

---

## 📊 BLOQUE 6 — Comparativa final de modelos

In [ ]:
# ─── Tabla comparativa ────────────────────────────────────────────────────────
resumen = pd.DataFrame([
    {
        'Modelo'        : nombre,
        'Accuracy Test' : round(v['acc_test'], 4),
        'CV Mean'       : round(v['cv_mean'],  4),
        'CV Std'        : round(v['cv_std'],   4),
        'Gap CV-Test'   : round(v['cv_mean'] - v['acc_test'], 4)
    }
    for nombre, v in resultados.items()
]).sort_values('Accuracy Test', ascending=False).reset_index(drop=True)

print('\n📊 COMPARATIVA DE MODELOS')
print('=' * 70)
print(resumen.to_string(index=False))
print(f'\n🏆 Mejor modelo: {resumen.iloc[0]["Modelo"]}')
print(f'   Gap CV-Test positivo = CV > Test → ligero overfitting')
print(f'   Gap CV-Test negativo = Test > CV → no hay overfitting')

In [ ]:
# ─── Gráfico comparativo ──────────────────────────────────────────────────────
os.makedirs('../reports', exist_ok=True)

colores = ['gold' if i == 0 else 'steelblue' for i in range(len(resumen))]
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy test
axes[0].barh(resumen['Modelo'], resumen['Accuracy Test'], color=colores)
axes[0].set_xlabel('Accuracy Test')
axes[0].set_title('Accuracy en Test por Modelo')
axes[0].set_xlim(0.5, 1.05)
for i, v in enumerate(resumen['Accuracy Test']):
    axes[0].text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=9)

# CV Mean ± Std
axes[1].barh(resumen['Modelo'], resumen['CV Mean'],
             xerr=resumen['CV Std'], color=colores, capsize=4)
axes[1].set_xlabel('CV Accuracy (mean ± std)')
axes[1].set_title('Cross-Validation 5-Fold')
axes[1].set_xlim(0.5, 1.05)

plt.suptitle('Comparativa de Modelos — p7_g1_multiclase', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../reports/comparativa_modelos.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Guardado: ../reports/comparativa_modelos.png')

---

## 🔬 BLOQUE 7 — Análisis detallado del mejor modelo

In [ ]:
# ─── Seleccionar mejor modelo automáticamente ─────────────────────────────────
nombre_ganador = resumen.iloc[0]['Modelo']
mejor          = resultados[nombre_ganador]
y_pred_mejor   = mejor['y_pred']

print(f'🏆 Modelo ganador : {nombre_ganador}')
print(f'   Accuracy test  : {mejor["acc_test"]:.4f}')
print(f'   CV mean        : {mejor["cv_mean"]:.4f} ± {mejor["cv_std"]:.4f}')

In [ ]:
# ─── Matriz de confusión (absoluta + normalizada) ─────────────────────────────
cm      = confusion_matrix(y_test, y_pred_mejor)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    ax=axes[0], colorbar=False, xticks_rotation=45)
axes[0].set_title(f'{nombre_ganador}\nMatriz de confusión (absoluta)')

ConfusionMatrixDisplay(cm_norm.round(2), display_labels=class_names).plot(
    ax=axes[1], colorbar=False, xticks_rotation=45)
axes[1].set_title(f'{nombre_ganador}\nMatriz de confusión (normalizada por fila)')

plt.tight_layout()
plt.savefig('../reports/confusion_matrix_best.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Guardado: ../reports/confusion_matrix_best.png')

In [ ]:
# ─── Classification report detallado como DataFrame ───────────────────────────
report_df = pd.DataFrame(
    classification_report(
        y_test, y_pred_mejor,
        target_names=class_names,
        output_dict=True
    )
).T.round(4)

print(f'\n📋 Classification Report — {nombre_ganador}')
print('=' * 65)
print(report_df)

# ─── Análisis de errores por clase ────────────────────────────────────────────
print('\n🔍 Clases con peor recall (posibles confusiones):')
print(report_df.loc[class_names, 'recall'].sort_values().head(3))

In [ ]:
# ─── Feature importance (si el modelo lo soporta) ─────────────────────────────
modelo_obj = mejor['modelo']

if hasattr(modelo_obj, 'feature_importances_'):
    imp = pd.Series(
        modelo_obj.feature_importances_,
        index=X_train.columns
    ).sort_values(ascending=True)

    plt.figure(figsize=(8, 6))
    imp.plot(kind='barh', color='steelblue')
    plt.title(f'{nombre_ganador} — Feature Importance')
    plt.xlabel('Importancia')
    plt.tight_layout()
    plt.savefig('../reports/feature_importance_best.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Guardado: ../reports/feature_importance_best.png')
else:
    print(f'ℹ️  {nombre_ganador} no expone feature_importances_ directamente.')
    print('   Para SVM o KNN considera SHAP (pip install shap) como alternativa.')

---

## 💾 BLOQUE 8 — Guardar modelo ganador y artefactos

In [ ]:
# ─── Guardar modelo ganador, resumen y predicciones ───────────────────────────
os.makedirs('../models',  exist_ok=True)
os.makedirs('../reports', exist_ok=True)

# Modelo
joblib.dump(mejor['modelo'], '../models/best_model.pkl')

# Tabla comparativa
resumen.to_csv('../reports/comparativa_modelos.csv', index=False)

# Predicciones del mejor modelo con etiquetas legibles
pd.DataFrame({
    'y_true'      : y_test.values,
    'y_pred'      : y_pred_mejor,
    'y_true_label': le.inverse_transform(y_test.values),
    'y_pred_label': le.inverse_transform(y_pred_mejor),
    'correcto'    : (y_test.values == y_pred_mejor).astype(int)
}).to_csv('../reports/predicciones_test.csv', index=False)

print('✅ Artefactos guardados:')
print(f'   ../models/best_model.pkl')
print(f'   ../reports/comparativa_modelos.csv')
print(f'   ../reports/predicciones_test.csv')
print(f'   ../reports/comparativa_modelos.png')
print(f'   ../reports/confusion_matrix_best.png')
print()
print(f'🏆 MODELO GANADOR  : {nombre_ganador}')
print(f'   Accuracy test   : {mejor["acc_test"]:.4f}')
print(f'   CV (5-fold)     : {mejor["cv_mean"]:.4f} ± {mejor["cv_std"]:.4f}')